# Project: Train YOLOv8 trên Google Colab với dataset từ Roboflow

Notebook này gồm các phần đúng theo yêu cầu:
1. **Trình bày cách train model**
2. **Trình bày kết quả**
3. **Phân tích kết quả**
4. **Kế hoạch cải thiện / tuần sau**

> Bạn chỉ cần thay `ROBOFLOW_API_KEY`, `WORKSPACE`, `PROJECT`, `VERSION` theo dataset của bạn trên Roboflow.


## 0. Kiểm tra GPU trên Colab

Trên Colab, vào: **Runtime → Change runtime type → Hardware accelerator → GPU**.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Cập nhật đường dẫn đến folder DPL301c của bạn
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/DPL301c'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"Kết quả sẽ được lưu tại: {DRIVE_OUTPUT_DIR}")

Mounted at /content/drive
Kết quả sẽ được lưu tại: /content/drive/MyDrive/DPL301c


In [ ]:
!nvidia-smi

Wed Jun 24 09:16:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Cài thư viện cần thiết

Notebook dùng `ultralytics` để train YOLOv8 và `roboflow` để tải dataset trực tiếp.

In [ ]:
!pip install -q ultralytics roboflow

import os
import glob
from pathlib import Path
from ultralytics import YOLO

print("Setup done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 144.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup done!


## 2. Tải dataset từ Roboflow

Vào Roboflow project của bạn, chọn **Download Dataset → YOLOv8 → Show Download Code**, rồi thay thông tin bên dưới.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="CAUE5ZvN0FJDbsoi6moP")
project = rf.workspace("dngs-workspace-k3fml").project("vietnam-traffic-sign-altsi-durze")
version = project.version(3)
dataset = version.download("yolov8")

DATA_YAML = f"{dataset.location}/data.yaml"
print("Dataset location:", dataset.location)
print("data.yaml:", DATA_YAML)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Vietnam-Traffic-Sign-3 in yolov8:: 100%|██████████| 41797/41797 [00:12<00:00, 3252.25it/s] 


Dataset location: /content/Vietnam-Traffic-Sign-3
data.yaml: /content/Vietnam-Traffic-Sign-3/data.yaml


## 3. Kiểm tra cấu trúc dataset

Dataset YOLOv8 thường có dạng:

```text
dataset/
├── train/images, train/labels
├── valid/images, valid/labels
├── test/images, test/labels
└── data.yaml
```

In [ ]:
print("Train images:", len(glob.glob(f"{dataset.location}/train/images/*")))
print("Valid images:", len(glob.glob(f"{dataset.location}/valid/images/*")))
print("Test images :", len(glob.glob(f"{dataset.location}/test/images/*")))

!cat {DATA_YAML}

Train images: 18285
Valid images: 1742
Test images : 869
names:
- DP-135
- P-102
- P-103a
- P-103b
- P-103c
- P-104
- P-106a
- P-106b
- P-107a
- P-112
- P-115
- P-117
- P-123a
- P-123b
- P-124a
- P-124b
- P-124c
- P-127
- P-128
- P-130
- P-131a
- P-137
- P-245a
- R-301c
- R-301d
- R-301e
- R-302a
- R-302b
- R-303
- R-407a
- R-409
- R-425
- R-434
- S-509a
- W-201a
- W-201b
- W-202a
- W-202b
- W-203b
- W-203c
- W-205a
- W-205b
- W-205d
- W-207a
- W-207b
- W-207c
- W-208
- W-209
- W-210
- W-219
- W-224
- W-225
- W-227
- W-233
- W-235
- W-245a
nc: 56
roboflow:
  license: CC BY 4.0
  project: vietnam-traffic-sign-altsi-durze
  url: https://universe.roboflow.com/dngs-workspace-k3fml/vietnam-traffic-sign-altsi-durze/dataset/3
  version: 3
  workspace: dngs-workspace-k3fml
test: ../test/images
train: ../train/images
val: ../valid/images


## 4. Kỹ thuật 1: Chọn model YOLOv8 phù hợp

| Model | Ưu điểm | Khi nào dùng |
|---|---|---|
| `yolov8n.pt` | nhanh, nhẹ | dataset nhỏ, chạy real-time, mobile |
| `yolov8s.pt` | cân bằng tốc độ và độ chính xác | lựa chọn phổ biến |
| `yolov8m.pt` | chính xác hơn | cần accuracy cao, GPU tốt |

Notebook này mặc định dùng `yolov8n.pt` để train nhanh trên Colab.

In [ ]:
MODEL_NAME = "yolov8s.pt"

model = YOLO(MODEL_NAME)
model.info()

YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs


(129, 11166560, 0, 28.816844800000002)

## 5. Kỹ thuật 2: Train baseline model

Đây là lần train đầu tiên để lấy kết quả nền.

Tham số quan trọng:
- `imgsz`: kích thước ảnh đầu vào
- `epochs`: số vòng train
- `batch`: batch size
- `patience`: early stopping nếu model không cải thiện

In [ ]:
# Chạy tiếp (resume) hoặc train mới lưu vào folder DPL301c trên Drive
results = model.train(
    data=DATA_YAML,
    epochs=20,
    imgsz=640,
    batch=16,
    patience=10,
    workers=2,
    project=DRIVE_OUTPUT_DIR, # Lưu vào folder DPL301c
    name="yolov8_baseline",
    exist_ok=True,
    resume=True # Tự động chạy tiếp nếu có checkpoint trong folder
)

       1/20      3.76G      1.076      2.863      1.096         42        640: 100% ━━━━━━━━━━━━ 1143/1143 3.0it/s 6:17
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 55/55 3.0it/s 18.4s
                   all       1742       3392      0.697        0.6      0.656      0.519

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/20      4.54G     0.8962      1.006     0.9993         41        640: 100% ━━━━━━━━━━━━ 1143/1143 3.1it/s 6:12
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 55/55 3.1it/s 17.8s
                   all       1742       3392      0.871       0.72      0.826      0.667

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/20      4.54G     0.8507     0.7662     0.9772         32        640: 100% ━━━━━━━━━━━━ 1143/1143 3.1it/s 6:05
                 Class     Images  Instances    

## 7. Kỹ thuật 4: Fine-tune với learning rate nhỏ hơn

Sau khi train xong, tiếp tục fine-tune từ model tốt nhất với learning rate nhỏ hơn để giảm loss ổn định hơn và cải thiện mAP.

In [ ]:
BEST_AUG_MODEL = "runs_project/yolov8_augmented/weights/best.pt"

model_finetune = YOLO(BEST_AUG_MODEL)

results_finetune = model_finetune.train(
    data=DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    workers=2,
    lr0=0.001,
    lrf=0.01,
    mosaic=0.3,
    project="runs_project",
    name="yolov8_finetune",
    exist_ok=True
)

## 8. Trình bày kết quả validation

| Chỉ số | Ý nghĩa |
|---|---|
| Precision | Trong các object model dự đoán, bao nhiêu cái đúng |
| Recall | Trong các object thật, model tìm được bao nhiêu |
| mAP50 | độ chính xác trung bình ở IoU 0.5 |
| mAP50-95 | chỉ số khắt khe hơn, trung bình từ IoU 0.5 đến 0.95 |

`mAP50-95` càng cao thì model càng tốt.

In [ ]:
BEST_MODEL = "runs_project/yolov8_finetune/weights/best.pt"

trained_model = YOLO(BEST_MODEL)
metrics = trained_model.val(data=DATA_YAML, imgsz=640, split="val")

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## 9. Hiển thị biểu đồ kết quả train

YOLOv8 tự lưu các hình:
- `results.png`: loss, precision, recall, mAP
- `confusion_matrix.png`: ma trận nhầm lẫn
- `PR_curve.png`: precision-recall curve
- `F1_curve.png`: F1 curve

In [ ]:
from IPython.display import Image, display

run_dir = "runs_project/yolov8_finetune"

for file in ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    path = os.path.join(run_dir, file)
    if os.path.exists(path):
        print("", file)
        display(Image(filename=path, width=900))
    else:
        print("Không tìm thấy:", path)

## 10. Test model trên ảnh test

Phần này dùng model đã train để dự đoán trên tập test.

In [ ]:
test_images = glob.glob(f"{dataset.location}/test/images/*")

print("Số ảnh test:", len(test_images))
print("Ví dụ ảnh test:", test_images[:3])

In [ ]:
predict_results = trained_model.predict(
    source=f"{dataset.location}/test/images",
    imgsz=640,
    conf=0.25,
    save=True,
    project="runs_project",
    name="test_predictions",
    exist_ok=True
)

print("Kết quả dự đoán được lưu tại: runs_project/test_predictions")

In [ ]:
pred_dir = "runs_project/test_predictions"
pred_imgs = glob.glob(f"{pred_dir}/*")

for img_path in pred_imgs[:10]:
    display(Image(filename=img_path, width=700))

## 11. Phân tích kết quả

### 11.1 Nhận xét chung
Model YOLOv8 đã học được đặc trưng của đối tượng trong dataset. Nếu `mAP50` cao nhưng `mAP50-95` thấp, nghĩa là model nhận ra object khá tốt nhưng bounding box chưa thật sự chính xác.

### 11.2 Phân tích Precision và Recall
- Precision cao: model ít dự đoán sai.
- Recall cao: model ít bỏ sót object.
- Precision thấp: model hay nhận nhầm background thành object.
- Recall thấp: model bỏ sót nhiều object thật.

### 11.3 Phân tích loss
- `box_loss` giảm: model học bounding box tốt hơn.
- `cls_loss` giảm: model phân loại class tốt hơn.
- `dfl_loss` giảm: model định vị box ổn định hơn.
- Nếu train loss giảm nhưng val loss tăng, model có thể bị overfitting.

### 11.4 Phân tích confusion matrix
Dùng confusion matrix để xem class nào bị nhầm nhiều nhất. Nếu một class bị nhầm nhiều, có thể do ảnh quá giống class khác, thiếu dữ liệu, label chưa chính xác, object quá nhỏ hoặc bị che khuất.

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "Metric": ["Precision", "Recall", "mAP50", "mAP50-95"],
    "Value": [
        float(metrics.box.mp),
        float(metrics.box.mr),
        float(metrics.box.map50),
        float(metrics.box.map)
    ],
    "Meaning": [
        "Tỷ lệ dự đoán đúng trong các object model phát hiện",
        "Tỷ lệ object thật được model phát hiện",
        "Độ chính xác trung bình tại IoU 0.5",
        "Độ chính xác trung bình từ IoU 0.5 đến 0.95"
    ]
})

summary

## 12. Xuất model

YOLOv8 có thể export sang nhiều định dạng:
- `.pt`: dùng lại trong Python
- `onnx`: dùng với ONNX Runtime
- `tflite`: dùng trên Android/mobile
- `ncnn`: dùng trên mobile/edge device

Nếu bạn làm app Android, có thể thử export TFLite.

In [ ]:
exported_path = trained_model.export(format="tflite", imgsz=640)
print("Exported model:", exported_path)